# Figure 1 — overview of the extracellular antigenic space

Length, pairwise similarity, how many distinct bins exist, and how much of the linear epitope space is private to a single receptor. All four extracellular segments, N-terminus included.

Needs `results/` populated: run `python run_all.py` first.

## Setup

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import style

style.apply()
if not style.available():
    print("Arial not found, matplotlib fell back to DejaVu Sans")

RESULTS, FIGS = ROOT / "results", ROOT / "figures"
FIGS.mkdir(exist_ok=True)
PAL = style.PALETTE
SEGMENTS = ["N-term", "ECL1", "ECL2", "ECL3"]

In [ ]:
def strip(ax, groups, values_by_group, log=False):
    """Jittered points with a median bar. Used in several panels."""
    for i, g in enumerate(groups):
        x = np.asarray(values_by_group[g], dtype=float)
        x = x[~np.isnan(x)]
        if not len(x):
            continue
        ax.scatter(np.random.normal(i, 0.09, len(x)), x, s=6, alpha=0.45,
                   color=PAL.get(g, "0.4"), edgecolors="none")
        ax.hlines(np.median(x), i - 0.28, i + 0.28, color="k", lw=1.6)
    ax.set_xticks(range(len(groups)))
    ax.set_xticklabels(groups)
    if log:
        ax.set_yscale("log")

In [ ]:
def pid_matrix(seg):
    return pd.read_csv(RESULTS / f"pid_{seg}.csv.gz", index_col=0)


def pid_values(seg):
    v = pid_matrix(seg).values.astype(float)
    iu = np.triu_indices_from(v, k=1)
    return v[iu][~np.isnan(v[iu])]

## Load

In [ ]:
seg = pd.read_csv(RESULTS / "segments_long.csv")
cc = pd.read_csv(RESULTS / "cluster_counts.csv")
km = pd.read_csv(RESULTS / "kmer_per_receptor.csv")
K = int(km["k"].max())
seg.head()

## Panels

In [ ]:
fig, ax = plt.subplot_mosaic([["len", "ecdf"], ["clust", "kmer"]],
                             figsize=(10.5, 7.8), constrained_layout=True)

strip(ax["len"], SEGMENTS,
      {s: seg.loc[seg.segment == s, "length"].values for s in SEGMENTS}, log=True)
ax["len"].set_ylabel("length (aa)")
ax["len"].set_title("Segment length")

a = ax["ecdf"]
for s in SEGMENTS:
    v = np.sort(pid_values(s))
    a.plot(v, np.arange(1, len(v) + 1) / len(v), color=PAL[s], label=s)
a.axvline(0.70, color="0.5", ls="--", lw=1)
a.set_xlabel("pairwise identity (shorter segment)")
a.set_ylabel("ECDF of pairs")
a.set_title("Pairwise similarity")
a.legend()

a = ax["clust"]
for s in SEGMENTS:
    d = cc[cc.segment == s].sort_values("identity_threshold")
    a.plot(d["identity_threshold"], d["n_clusters"], "o-", ms=4, color=PAL[s], label=s)
a.invert_xaxis()
a.set_xlabel("identity threshold")
a.set_ylabel("n clusters")
a.set_title("Distinct bins")
a.legend()

d = km[km.k == K]
strip(ax["kmer"], SEGMENTS,
      {s: d.loc[d.segment == s, "frac_private"].values for s in SEGMENTS})
ax["kmer"].set_ylabel(f"fraction of {K}-mers unique to that receptor")
ax["kmer"].set_title(f"Private linear epitope space (k={K})")

fig.savefig(FIGS / "01_overview.png")